# Phase 6 — Real-World Fine-Tuning v3 (Continuous Learning)
**Notebook độc lập** — chạy sau khi đã thu thập DataRealTest từ testPC.py

## Chiến lược dữ liệu (v3 — Dual Source + Per-Class Oversample + KD)
```
[OLD Dataset v1]  → 100% → split 80/20 → train_old / val_old (4,497)
[REAL Dataset]    → 90% train / 10% real_val (per-class, min 1)
                    Per-class adaptive oversample → train_real_virtual
                    real_val chỉ LOG, không dùng early stopping
                                 ↓
  TRAIN = train_old + train_real_virtual
  VAL   = val_old (ổn định, đánh giá khách quan)
  LOG   = real_val (monitor domain adaptation thực tế)
                                 ↓
  Phase 6: LR siêu nhỏ + KD Loss (T=4) + clip_grad=1.0
                                 ↓
  best_model_p6.pth  →  export ONNX v3  →  deploy testPC.py
```

## Cải tiến v3 so với v2
| # | Vấn đề v2 | Giải pháp v3 |
|---|---|---|
| 1 | `p6_old_ratio=0.90` lãng phí ~2k ảnh | `p6_old_ratio=1.0` — dùng toàn bộ old data |
| 2 | Oversample đồng đều ×30 | Per-class adaptive — class ít ảnh được boost mạnh hơn |
| 3 | Không có Knowledge Distillation | KD Loss từ frozen Phase 5 teacher (KL divergence, T=4) |
| 4 | `clip_grad_norm_=5.0` quá cao cho phase 6 | `clip_grad_norm_=1.0` — bảo vệ backbone |
| 5 | Không quan sát real-world accuracy | Mini `real_val` log song song (không ảnh hưởng early stop) |
| 6 | EMA decay=0.9998 phản ứng chậm | EMA decay=0.9995 — bắt kịp cải thiện nhanh hơn |

> **TRƯỚC KHI CHẠY:**
> - **Cell 7**: Đổi `DATA_DIR_OLD` và `DATA_DIR_REAL`
> - **Cell 2**: Đổi `p6_base_ckpt` trỏ đến file `best_model.pth` từ Phase 5


In [1]:
# ============================================================
# CELL 1 - IMPORTS & HARDWARE
# ============================================================
import os, sys, time, copy, json, math, gc, warnings, contextlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR
from torchvision import transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

%pip install tqdm -q
from tqdm import tqdm

warnings.filterwarnings('ignore')

print('\n' + '='*70)
print('  WASTE DETECTION — Phase 6 Real-World Fine-Tuning  [v3]')
print('='*70)

if torch.cuda.is_available():
    GPU_COUNT = torch.cuda.device_count()
    GPU_MEM   = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE    = 'cuda'
    torch.backends.cudnn.benchmark = True
    print(f'  CUDA! GPUs: {GPU_COUNT}')
    for i in range(GPU_COUNT):
        print(f'    GPU {i}: {torch.cuda.get_device_name(i)} '
              f'({torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    GPU_COUNT, GPU_MEM, DEVICE = 1, 0, 'mps'
    print('  Apple MPS')
else:
    GPU_COUNT, GPU_MEM, DEVICE = 0, 0, 'cpu'
    print('  CPU only')

if os.path.exists('/kaggle/input'):
    _found = None
    for root, dirs, _ in os.walk('/kaggle/input'):
        if any(d in dirs for d in ['Battery', 'battery', 'Biological', 'biological']):
            _found = root; break
    _DEFAULT_DATA = _found or '/kaggle/input'
    _DEFAULT_OUT  = '/kaggle/working/outputs_v3'
    PLATFORM = 'Kaggle'
else:
    _candidates = [
        './Model/DataSet/Data', './Model/DataSet',
        './DataSet/Data', './DataSet', './dataset', './data',
        '../Model/DataSet/Data', '../Model/DataSet',
        '../DataSet/Data', '../DataSet', '../dataset', '../data',
        '/Users/capkimkhanh/Downloads/DataSet',
    ]
    _DEFAULT_DATA = None
    for _p in _candidates:
        if os.path.isdir(_p):
            _DEFAULT_DATA = _p; break
    _DEFAULT_DATA = _DEFAULT_DATA or './Model/DataSet/Data'
    _DEFAULT_OUT  = './outputs_v3'
    if not os.path.isdir('./outputs_v2') and os.path.isdir('./Model/Train/outputs_v2'):
        _DEFAULT_OUT = './Model/Train/outputs_v3'
    PLATFORM = 'Local'

print(f'  Platform: {PLATFORM} | Data: {_DEFAULT_DATA}')

_SCALE = max(GPU_COUNT, 1)
if DEVICE == 'cuda' and GPU_MEM >= 14:
    _B = {'p1': 24, 'p2': 16, 'p3': 8, 'p4': 8}
elif DEVICE == 'cuda':
    _B = {'p1': 12, 'p2': 8,  'p3': 4, 'p4': 4}
elif DEVICE == 'mps':
    _B = {'p1': 20, 'p2': 10, 'p3': 8, 'p4': 8}
else:
    _B = {'p1': 4,  'p2': 4,  'p3': 2, 'p4': 2}

for ph, b in _B.items():
    print(f'  Batch {ph.upper()}: {b}/GPU -> {b*_SCALE} effective')

_IN_NOTEBOOK = 'ipykernel' in sys.modules
if PLATFORM == 'Local' and _IN_NOTEBOOK and DEVICE != 'cuda':
    _NUM_WORKERS = 0
elif PLATFORM == 'Local' and _IN_NOTEBOOK and DEVICE == 'cuda':
    _NUM_WORKERS = 2
else:
    _NUM_WORKERS = 2
print(f'  Workers: {_NUM_WORKERS} | Notebook: {_IN_NOTEBOOK}')

if DEVICE == 'mps':
    try:
        torch.mps.set_per_process_memory_fraction(0.72)
        print('  MPS memory fraction: 72%')
    except AttributeError:
        pass
print('='*70)


Note: you may need to restart the kernel to use updated packages.

  WASTE DETECTION — Phase 6 Real-World Fine-Tuning  [v3]
  Apple MPS
  Platform: Local | Data: ../DataSet/Data
  Batch P1: 20/GPU -> 20 effective
  Batch P2: 10/GPU -> 10 effective
  Batch P3: 8/GPU -> 8 effective
  Batch P4: 8/GPU -> 8 effective
  Workers: 0 | Notebook: True
  MPS memory fraction: 72%


In [2]:
# ============================================================
# CELL 2 - CONFIG & CLASSES  (v3)
# ============================================================

CLASSES = [
    'Battery', 'Biological', 'General_Waste',
    'Glass', 'Metal', 'Paper_Cardboard', 'Plastic',
]

CLASS_ALIASES = {
    'General Waste khẩu trang tàn thuốc găng tay': 'General_Waste',
    'General_Waste': 'General_Waste',
    'General Waste': 'General_Waste',
    'Paper+Cardboard gộp chung thành paper': 'Paper_Cardboard',
    'Paper+Cardboard': 'Paper_Cardboard',
    'Paper_Cardboard': 'Paper_Cardboard',
    'Paper': 'Paper_Cardboard',
}

def _norm_name(name):
    return name.strip().lower().replace('-', '_').replace(' ', '_')

CANONICAL_BY_NORM = {_norm_name(c): c for c in CLASSES}
for alias_name, canonical_name in CLASS_ALIASES.items():
    CANONICAL_BY_NORM[_norm_name(alias_name)] = canonical_name

NUM_CLASSES = len(CLASSES)
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}

if DEVICE == 'mps':
    _P2_IMG = 448
else:
    _P2_IMG = 540


_DEFAULT_P6_CKPT = 'outputs_v2/best_model_v2.pth'
if not os.path.exists(_DEFAULT_P6_CKPT) and os.path.exists('./Model/Train/outputs_v2/best_model_v2.pth'):
    _DEFAULT_P6_CKPT = './Model/Train/outputs_v2/best_model_v2.pth'

CONFIG = {
    'data_dir'             : _DEFAULT_DATA,
    'output_dir'           : _DEFAULT_OUT,
    'img_size'             : 384,
    'num_workers'          : _NUM_WORKERS,
    'device'               : DEVICE,
    'gpu_count'            : max(GPU_COUNT, 1),
    'seed'                 : 42,
    'val_split'            : 0.2,
    'weight_decay'         : 1e-4,
    'ema_decay'            : 0.9995,
    'focal_gamma'          : 2.0,
    'ema_eval_freq'        : 2,
    'sampler_power'        : 0.7,
    'alpha_power'          : 0.7,
    'min_samples_per_class': 20,

    # ── Adaptive Gamma Correction ─────────────────────────────────────────
    'agc_target'           : 128,
    'agc_gamma_min'        : 0.4,
    'agc_gamma_max'        : 3.0,
    'agc_jitter'           : 25,

    # Phase 1–5 (không thay đổi)
    'p1_epochs': 5,  'p1_lr': 1e-3, 'p1_img_size': 360,
    'p1_batch' : _B['p1']*_SCALE, 'p1_label_smooth': 0.0,
    'p1_mixup' : 0.0, 'p1_cutmix': 0.0, 'p1_aug': 'light',

    'p2_epochs': 12, 'p2_lr_head': 4e-4, 'p2_lr_backbone': 4e-5,
    'p2_img_size': _P2_IMG, 'p2_batch': _B['p2']*_SCALE,
    'p2_label_smooth': 0.05, 'p2_mixup': 0.1, 'p2_cutmix': 0.0, 'p2_aug': 'medium',

    'p3_epochs': 45, 'p3_lr_head': 2.5e-4, 'p3_lr_backbone': 8e-6,
    'p3_img_size': 384, 'p3_batch': _B['p3']*_SCALE,
    'p3_label_smooth': 0.08, 'p3_mixup': 0.2, 'p3_cutmix': 0.2,
    'p3_aug': 'heavy', 'p3_patience': 18,

    'p4_epochs': 12, 'p4_lr': 3e-5, 'p4_img_size': 384,
    'p4_batch': _B['p4']*_SCALE, 'p4_label_smooth': 0.03,
    'p4_mixup': 0.0, 'p4_cutmix': 0.0, 'p4_aug': 'medium',

    'p5_epochs': 15, 'p5_lr_head': 5e-5, 'p5_lr_backbone': 2e-6,
    'p5_img_size': 384, 'p5_label_smooth': 0.03,
    'p5_mixup': 0.1, 'p5_cutmix': 0.0, 'p5_aug': 'lighting',

    # ── Phase 6: Real-World Fine-Tuning v3 ──────────────────────────────
    # << ĐỔI path này trước khi chạy
    'p6_base_ckpt'      : _DEFAULT_P6_CKPT,

    'p6_epochs'         : 60,
    'p6_patience'       : 20,
    'p6_lr_head'        : 2e-5,
    'p6_lr_backbone'    : 6e-7,
    'p6_img_size'       : 384,
    'p6_batch'          : _B['p3'] * _SCALE,
    'p6_label_smooth'   : 0.02,
    'p6_mixup'          : 0.05,
    'p6_cutmix'         : 0.0,
    'p6_aug'            : 'lighting',
    'p6_val_ratio'      : 0.20,

    # [v3 FIX] EMA decay 0.9995 (v2: 0.9998)
    # Half-life: 0.9995 ≈ 1,386 steps (0.7 epoch) vs 0.9998 ≈ 3,465 steps (1.7 epoch)
    # → Phản ứng nhanh hơn với cải thiện từ real data trong 20 epochs ngắn
    'p6_ema_decay'      : 0.9995,

    # [v3 FIX] old_ratio = 1.0 (v2: 0.90)
    # Dùng 100% old data → nhiều data chống forgetting hơn.
    # Regularization nên đến từ dropout/aug, không phải bỏ data.
    'p6_old_ratio'      : 1.0,

    # [v3] Per-class adaptive oversample
    # target_per_class = max(real_counts_per_class) × factor
    # factor=8 cân bằng real domain mạnh hơn old dataset nhưng tránh lặp quá đà class ít ảnh.
    'p6_real_oversample_factor': 8,
    'p6_real_min_target'       : 1500,
    'p6_real_target_cap'       : 3500,

    # [v3] Real monitor split — chỉ dùng để đo domain thực tế, không early stop
    # Với >1k ảnh real hiện tại: 15% đủ lớn để theo dõi real accuracy theo từng class.
    # early stopping VẪN dùng val_old (ổn định)
    'p6_real_val_pct'   : 0.15,
    'p6_real_val_min_per_class': 5,

    # [v3] Chọn checkpoint theo mục tiêu deploy thực tế
    # 'real': lưu model có real_val_acc cao nhất; 'hybrid': kết hợp old/real.
    'p6_selection_metric': 'real',
    'p6_real_score_weight': 0.80,

    # [v3] Knowledge Distillation từ frozen Phase 5 teacher
    # KL(student_soft || teacher_soft) × T² — classic Hinton KD
    # T=4: soft targets đủ mềm để transfer "dark knowledge"
    # weight=0.3: total_loss = 0.7×focal + 0.3×kd
    #   → Tăng kd_weight nếu model quên old data nhanh
    #   → Giảm kd_weight nếu KD cản trở học real data mới
    'p6_kd_temperature' : 4.0,
    'p6_kd_weight'      : 0.3,

    # [v3 FIX] clip_grad = 1.0 (v2: 5.0)
    # Phase 6 dùng LR backbone=1e-6, nhưng gradient từ distribution shift
    # (real data khác old data) có thể bùng nổ cục bộ.
    # clip=1.0 bảo vệ backbone weights đã học tốt từ Phase 1-5.
    'p6_clip_grad'      : 1.0,
}

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG['seed'])
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print(f'Classes ({NUM_CLASSES}): {CLASSES}')
print(f'Output dir : {CONFIG["output_dir"]}')
print(f'AGC        : target={CONFIG["agc_target"]}  '
      f'clip=[{CONFIG["agc_gamma_min"]}, {CONFIG["agc_gamma_max"]}]  '
      f'jitter=±{CONFIG["agc_jitter"]}')
print()
print('Phase 6 v3 config:')
print(f'  base_ckpt  = {CONFIG["p6_base_ckpt"]}')
print(f'  epochs     = {CONFIG["p6_epochs"]}  patience={CONFIG["p6_patience"]}')
print(f'  LR head    = {CONFIG["p6_lr_head"]:.0e}  backbone={CONFIG["p6_lr_backbone"]:.0e}')
print(f'  EMA decay  = {CONFIG["p6_ema_decay"]}  (v2 was 0.9998)')
print(f'  old_ratio  = {CONFIG["p6_old_ratio"]}  (v2 was 0.90)')
print(f'  real target= factor {CONFIG["p6_real_oversample_factor"]}, min {CONFIG["p6_real_min_target"]}, cap {CONFIG["p6_real_target_cap"]}')
print(f'  KD T       = {CONFIG["p6_kd_temperature"]}  weight={CONFIG["p6_kd_weight"]}')
print(f'  clip_grad  = {CONFIG["p6_clip_grad"]}  (v2 was 5.0)')
print(f'  real_val % = {CONFIG["p6_real_val_pct"]}  min/class={CONFIG["p6_real_val_min_per_class"]}')
print(f'  selection  = {CONFIG["p6_selection_metric"]}  real_weight={CONFIG["p6_real_score_weight"]}')


Classes (7): ['Battery', 'Biological', 'General_Waste', 'Glass', 'Metal', 'Paper_Cardboard', 'Plastic']
Output dir : ./outputs_v3
AGC        : target=128  clip=[0.4, 3.0]  jitter=±25

Phase 6 v3 config:
  base_ckpt  = outputs_v2/best_model_v2.pth
  epochs     = 60  patience=20
  LR head    = 2e-05  backbone=6e-07
  EMA decay  = 0.9995  (v2 was 0.9998)
  old_ratio  = 1.0  (v2 was 0.90)
  real target= factor 8, min 1500, cap 3500
  KD T       = 4.0  weight=0.3
  clip_grad  = 1.0  (v2 was 5.0)
  real_val % = 0.15  min/class=5
  selection  = real  real_weight=0.8


In [3]:
# CELL 3 - DATASET & AUGMENTATION  (v5 - Adaptive Gamma)
# ============================================================
import math as _math

# ══════════════════════════════════════════════════════════════════════════════
#  ADAPTIVE GAMMA CORRECTION  —  Giải thích đầy đủ
# ══════════════════════════════════════════════════════════════════════════════
#
#  Vấn đề với gamma cố định (v4):
#    - Gamma = 1.2 kéo sáng MỌI ảnh như nhau, kể cả ảnh đã sáng đủ rồi.
#    - Ánh sáng mặt trời buổi trưa (mean≈200) → ảnh càng bị chói hơn.
#    - Đêm tối (mean≈30) → 1.2 chưa đủ để kéo sáng.
#
#  Giải pháp Adaptive Gamma:
#    Bước 1: Đo độ sáng trung bình của ảnh (kênh V trong không gian màu HSV).
#            Kênh V = max(R,G,B) / 255 — đây là luminance perceptual chính xác nhất,
#            không bị ảnh hưởng bởi màu sắc (hue/saturation).
#
#    Bước 2: Tính gamma cần thiết để đưa mean_V về target (mặc định 128):
#              γ = log(target/255) / log(mean_V/255)
#            Nếu mean_V < target → log(mean_V/255) âm hơn log(target/255)
#              → γ < 1 → ảnh được kéo SÁNG (đúng, vì ảnh đang tối).
#            Nếu mean_V > target → γ > 1 → ảnh được DÌM (đúng, vì ảnh đang chói).
#
#    Bước 3: Build Lookup Table (256 giá trị) và apply — O(1), vài micro-giây.
#
#    Bước 4 (chỉ training): Jitter target ± N để model học robustness với
#            các mức sáng hơi khác nhau thay vì overfit vào đúng target=128.
#
#  Sơ đồ các trường hợp:
#    mean=30  (đêm)    → γ≈0.45  → kéo sáng mạnh
#    mean=80  (trong nhà tối) → γ≈0.78  → kéo sáng nhẹ
#    mean=128 (lý tưởng)  → γ=1.00  → pass-through
#    mean=180 (nắng)   → γ≈1.65  → dìm sáng vừa
#    mean=220 (nắng gắt) → γ≈2.40  → dìm sáng mạnh
# ══════════════════════════════════════════════════════════════════════════════

class AdaptiveGammaCorrection:
    """
    Adaptive Gamma Correction động theo từng ảnh.

    Dùng cho VALIDATION và INFERENCE (target cố định → kết quả deterministic).

    Args:
        target      : độ sáng chuẩn muốn normalize về (0-255), mặc định 128.
        gamma_min   : clamp dưới — giới hạn kéo sáng tối đa (mặc định 0.4).
        gamma_max   : clamp trên — giới hạn dìm sáng tối đa (mặc định 3.0).
    """
    def __init__(self, target: int = 128,
                 gamma_min: float = 0.4,
                 gamma_max: float = 3.0):
        self.target    = int(target)
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max
        # Index array dùng để build LUT nhanh
        self._idx = np.arange(256, dtype=np.float64) / 255.0

    def _compute_gamma(self, mean_v: float) -> float:
        """
        Công thức: γ = log(target/255) / log(mean_V/255)

        Edge cases:
          - mean_v quá nhỏ (<8): clamp để tránh gamma = ∞
          - mean_v quá lớn (>247): clamp để tránh division by zero
          - mean_v xấp xỉ target: trả về 1.0 (không cần sửa)
        """
        mean_v  = float(np.clip(mean_v, 8.0, 247.0))
        target  = float(np.clip(self.target, 8.0, 247.0))

        log_mean   = _math.log(mean_v   / 255.0)
        log_target = _math.log(target   / 255.0)

        # Nếu mean gần target (sai lệch < 3%): không cần sửa
        if abs(log_mean - log_target) < 0.03:
            return 1.0

        gamma = log_target / log_mean
        return float(np.clip(gamma, self.gamma_min, self.gamma_max))

    def _build_lut(self, gamma: float) -> np.ndarray:
        """Build Lookup Table 256 giá trị dựa trên gamma."""
        lut = np.power(self._idx, gamma) * 255.0
        return lut.clip(0, 255).astype(np.uint8)

    def _get_mean_v(self, arr: np.ndarray) -> float:
        """
        Tính độ sáng trung bình trên kênh V (Value) của HSV.
        V = max(R, G, B) — đây là luminance perceptual chính xác.
        Nhanh hơn convert PIL sang HSV và không cần import thêm thư viện.
        """
        return arr.max(axis=2).mean().item()

    def compute_gamma_for_image(self, img: Image.Image) -> float:
        """Public method để debug / log gamma của từng ảnh."""
        arr = np.asarray(img, dtype=np.uint8)
        return self._compute_gamma(self._get_mean_v(arr))

    def __call__(self, img: Image.Image) -> Image.Image:
        arr    = np.asarray(img, dtype=np.uint8)
        mean_v = self._get_mean_v(arr)
        gamma  = self._compute_gamma(mean_v)

        if abs(gamma - 1.0) < 0.02:
            return img                         # pass-through, không cần xử lý

        lut = self._build_lut(gamma)
        out = lut[arr]                         # vectorised index — cực nhanh O(1)
        return Image.fromarray(out.astype(np.uint8))

    def __repr__(self):
        return (f'AdaptiveGammaCorrection(target={self.target}, '
                f'clip=[{self.gamma_min}, {self.gamma_max}])')


class RandomAdaptiveGammaJitter:
    """
    Phiên bản Training của Adaptive Gamma Correction.

    Thêm jitter ngẫu nhiên vào target brightness để model học robustness:
      target_actual = target_base ± uniform(0, jitter)
      → [target - jitter, target + jitter]

    Tại sao cần jitter?
      Nếu val/inference luôn normalize về ĐÚNG 128, mà training cũng về 128,
      model sẽ overfit: "ảnh nào cũng có mean=128 → chỉ cần học màu sắc/hình dạng".
      Với jitter=25, model thấy ảnh có mean từ 103-153 → học robust hơn.
    """
    def __init__(self, target: int = 128, jitter: int = 25,
                 gamma_min: float = 0.4, gamma_max: float = 3.0,
                 p: float = 0.85):
        self.target    = target
        self.jitter    = jitter
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max
        self.p         = p     # xác suất áp dụng (để 15% ảnh không bị sửa)
        self._base_agc = AdaptiveGammaCorrection(target, gamma_min, gamma_max)
        self._idx      = np.arange(256, dtype=np.float64) / 255.0

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        # Jitter target
        jittered_target = int(np.clip(
            self.target + np.random.randint(-self.jitter, self.jitter + 1),
            max(self.target - self.jitter, 10),
            min(self.target + self.jitter, 245),
        ))
        agc = AdaptiveGammaCorrection(jittered_target, self.gamma_min, self.gamma_max)
        return agc(img)

    def __repr__(self):
        return (f'RandomAdaptiveGammaJitter(target={self.target}±{self.jitter}, '
                f'p={self.p})')


# ── Lighting Augmentation Transforms ────────────────────────────────────────

class RandomShadowInjection:
    """
    Mô phỏng bóng đổ cứng (hard shadow) — bóng cành cây / tay người / thanh sắt.
    Tạo vệt đen hình chữ nhật ngẫu nhiên theo chiều ngang hoặc dọc.

    Quan trọng: Áp dụng SAU AdaptiveGamma (shadow là nhiễu residual sau normalize).
    """
    def __init__(self, p: float = 0.4, dark_lo: float = 0.3, dark_hi: float = 0.65):
        self.p       = p
        self.dark_lo = dark_lo
        self.dark_hi = dark_hi

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape[:2]

        if np.random.rand() < 0.5:
            y1 = np.random.randint(0, max(1, h // 2))
            y2 = np.random.randint(h // 2, h)
            x1, x2 = 0, w
        else:
            x1 = np.random.randint(0, max(1, w // 2))
            x2 = np.random.randint(w // 2, w)
            y1, y2 = 0, h

        factor = np.random.uniform(self.dark_lo, self.dark_hi)
        arr[y1:y2, x1:x2] *= factor
        return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

    def __repr__(self):
        return f'RandomShadowInjection(p={self.p})'


class RandomHardLight:
    """
    Mô phỏng ánh sáng chói cục bộ (hard light flare) — nắng gắt chiếu thẳng
    hoặc đèn đường gần camera Raspberry Pi.

    Tạo vùng sáng ellipse với Gaussian mask viền mềm.
    Áp dụng SAU AdaptiveGamma — đây là nhiễu residual không normalize được.
    """
    def __init__(self, p: float = 0.3, intensity_lo: float = 1.4, intensity_hi: float = 2.5):
        self.p            = p
        self.intensity_lo = intensity_lo
        self.intensity_hi = intensity_hi

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img

        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape[:2]
        cx = np.random.randint(w // 4, 3 * w // 4)
        cy = np.random.randint(h // 4, 3 * h // 4)
        rx = np.random.randint(w // 6, w // 3)
        ry = np.random.randint(h // 6, h // 3)

        yy, xx = np.ogrid[:h, :w]
        mask = np.exp(-(((xx - cx) / max(rx, 1)) ** 2 + ((yy - cy) / max(ry, 1)) ** 2))
        intensity = np.random.uniform(self.intensity_lo, self.intensity_hi)
        arr += mask[:, :, np.newaxis] * intensity * 60
        return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

    def __repr__(self):
        return f'RandomHardLight(p={self.p})'


def load_rgb_on_white(path, bg=(255, 255, 255)):
    """Load ảnh RGB; nếu PNG có alpha thì composite lên nền trắng như môi trường thực tế."""
    with Image.open(path) as img:
        has_alpha = img.mode in ('RGBA', 'LA') or ('transparency' in img.info)
        if has_alpha:
            rgba = img.convert('RGBA')
            canvas = Image.new('RGBA', rgba.size, (*bg, 255))
            canvas.alpha_composite(rgba)
            return canvas.convert('RGB')
        return img.convert('RGB')


class RandomWhiteBackgroundJitter:
    """Làm nền trắng hơi lệch sáng/nhiễu nhẹ để model không học thuộc viền remove-background."""
    def __init__(self, p=0.35, threshold=242, value_lo=235, value_hi=255, noise=3):
        self.p = p
        self.threshold = threshold
        self.value_lo = value_lo
        self.value_hi = value_hi
        self.noise = noise

    def __call__(self, img):
        if np.random.rand() > self.p:
            return img
        arr = np.array(img).copy()
        mask = (arr[:, :, 0] >= self.threshold) & \
               (arr[:, :, 1] >= self.threshold) & \
               (arr[:, :, 2] >= self.threshold)
        if not mask.any():
            return img
        base = np.random.randint(self.value_lo, self.value_hi + 1, size=(1, 1, 3))
        if self.noise > 0:
            jitter = np.random.randint(-self.noise, self.noise + 1, size=arr.shape)
            fill = np.clip(base + jitter, 0, 255).astype(np.uint8)
        else:
            fill = np.broadcast_to(base.astype(np.uint8), arr.shape)
        arr[mask] = fill[mask]
        return Image.fromarray(arr)

    def __repr__(self):
        return f'RandomWhiteBackgroundJitter(p={self.p})'


# ── Dataset ──────────────────────────────────────────────────────────────────

class WasteDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = load_rgb_on_white(path)
        if self.transform:
            img = self.transform(img)
        return img, label


def _resolve_data_root(data_dir):
    data_path = Path(data_dir)
    fallback_candidates = [
        Path('./Model/DataSet/Data'), Path('./Model/DataSet'),
        Path('./DataSet/Data'), Path('./DataSet'),
        Path('./dataset'), Path('./data'),
    ]
    if not data_path.exists():
        for p in fallback_candidates:
            if p.exists():
                data_path = p; break
    if not data_path.exists():
        raise FileNotFoundError(f'Data dir not found: {data_dir}')
    nested = data_path / 'Data'
    if nested.exists() and nested.is_dir():
        data_path = nested
    return data_path


def _canonical_from_folder(folder_name):
    return CANONICAL_BY_NORM.get(_norm_name(folder_name))


def load_samples(data_dir):
    samples   = []
    data_path = _resolve_data_root(data_dir)
    print(f'[INFO] Data root: {data_path}')

    folder_map       = {f.name: f for f in data_path.iterdir() if f.is_dir()}
    canonical_to_dir = {}
    for fn, fp in folder_map.items():
        cn = _canonical_from_folder(fn)
        if cn and cn not in canonical_to_dir:
            canonical_to_dir[cn] = fp
    for cls_name in CLASSES:
        exact = data_path / cls_name
        if exact.exists() and exact.is_dir():
            canonical_to_dir[cls_name] = exact

    missing = [c for c in CLASSES if c not in canonical_to_dir]
    if missing:
        raise RuntimeError(f'Missing class folders: {missing}')

    for cls_name in CLASSES:
        cls_dir = canonical_to_dir[cls_name]; cls_idx = CLASS2IDX[cls_name]; cnt = 0
        for ext in ('*.jpg','*.jpeg','*.png','*.bmp','*.webp',
                    '*.JPG','*.JPEG','*.PNG','*.BMP','*.WEBP'):
            for p in cls_dir.glob(ext):
                samples.append((str(p), cls_idx)); cnt += 1
        print(f'  {cls_name:<20}: {cnt} images')

    print(f'  Total: {len(samples)}')
    if samples:
        labels = np.array([s[1] for s in samples])
        binc   = np.bincount(labels, minlength=NUM_CLASSES)
        print(f'  Imbalance ratio: {(binc.max()/max(binc.min(),1)):.2f}x')
    return samples


def train_val_split_stratified(samples, val_ratio=0.2, seed=42):
    labels = np.array([s[1] for s in samples])
    uniq, cnts = np.unique(labels, return_counts=True)
    if len(uniq) != NUM_CLASSES:
        raise RuntimeError(f'Found {len(uniq)} classes, expected {NUM_CLASSES}.')
    if np.any(cnts < 2):
        print('[WARN] Some classes <2 — random split.')
        rng = np.random.default_rng(seed); idx = np.arange(len(samples)); rng.shuffle(idx)
        val_n = min(max(NUM_CLASSES, int(len(samples)*val_ratio)), len(samples)-1)
        return [samples[i] for i in idx[val_n:]], [samples[i] for i in idx[:val_n]]
    eff = max(val_ratio, NUM_CLASSES/len(samples))
    sss = StratifiedShuffleSplit(1, test_size=eff, random_state=seed)
    ti, vi = next(sss.split(np.zeros(len(samples)), labels))
    return [samples[i] for i in ti], [samples[i] for i in vi]


# ── Transform Factory ─────────────────────────────────────────────────────────

def get_transforms(img_size: int, aug_level: str = 'light', config: dict = None):
    """
    Tạo transform pipeline — v7: RandomResizedCrop cho training.

    Chiến lược:
      Training (light/medium/heavy/lighting):
        AGC jitter → RandomResizedCrop(scale, ratio) → augment → Normalize
        - AGC TRƯỚC crop để đo mean brightness trên toàn ảnh gốc (chính xác hơn).
        - RandomResizedCrop một mình thay thế Resize+RandomCrop:
            crop ngẫu nhiên 40-100% diện tích ảnh, từ BẤT KỲ vị trí nào,
            resize về img_size với BICUBIC — không méo aspect ratio của object.
        - Object ở góc? ✓  Object ở viền? ✓  Object ở giữa? ✓
        - Tiêu chuẩn của Inception, EfficientNet, MobileNet, ResNet.

      Validation / Inference:
        Resize(int, ngắn nhất) → AGC deterministic → CenterCrop → Normalize
        - Tại sao CenterCrop? Val pipeline mirror INFERENCE thực tế:
            Camera Raspberry Pi CỐ ĐỊNH → thùng rác luôn ở trung tâm frame.
            Val đo accuracy ở điều kiện inference, không phải dataset distribution.
        - CenterCrop đúng cho use case này.

    aug_level:
      'val'      — Resize(int) + AGC + CenterCrop  [val & inference]
      'light'    — AGC + RandomResizedCrop(60-100%) + flip nhẹ
      'medium'   — AGC + RandomResizedCrop(50-100%) + augment vừa
      'heavy'    — AGC + RandomResizedCrop(40-100%) + full lighting aug
      'lighting' — AGC + RandomResizedCrop(50-100%) + lighting cường độ cao
    """
    cfg = config or CONFIG
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    agc_target   = cfg.get('agc_target',    128)
    agc_min      = cfg.get('agc_gamma_min', 0.4)
    agc_max      = cfg.get('agc_gamma_max', 3.0)
    agc_jitter   = cfg.get('agc_jitter',    25)

    # AGC deterministic — dùng cho val/inference
    agc_det = AdaptiveGammaCorrection(agc_target, agc_min, agc_max)
    # AGC với jitter — dùng cho training
    agc_jit = RandomAdaptiveGammaJitter(agc_target, agc_jitter, agc_min, agc_max, p=0.85)

    # Kích thước resize cạnh ngắn cho val (margin 15% — chuẩn ImageNet)
    val_resize = int(img_size * 1.15)

    # ── Validation / Inference ────────────────────────────────────────────
    # Tại sao vẫn dùng CenterCrop cho val/inference?
    #   Val pipeline phải mirror INFERENCE thực tế — không phải dataset distribution.
    #   Camera Raspberry Pi được gắn cố định hướng vào thùng rác
    #   → object luôn nằm ở trung tâm frame khi inference thực tế.
    #   → CenterCrop đúng cho inference.
    #   → Val pipeline dùng CenterCrop để đo accuracy đúng với điều kiện thực.
    #
    # Resize(int) giữ aspect ratio → ảnh không bị méo trước khi crop.
    # val_resize = img_size * 1.15 để có thêm padding xung quanh trước CenterCrop.
    if aug_level == 'val':
        return transforms.Compose([
            transforms.Resize(val_resize),       # << int → giữ aspect ratio
            agc_det,                             # << AGC trước crop (đo mean toàn ảnh)
            transforms.CenterCrop(img_size),     # << camera Pi cố định → đúng
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    # ── Light (Phase 1 - Head warm-up) ───────────────────────────────────
    # RandomResizedCrop thay thế Resize + RandomCrop:
    #   - Crop ngẫu nhiên vùng 60-100% diện tích ảnh, từ BẤT KỲ vị trí nào
    #   - Resize vùng crop về img_size (không méo aspect ratio của object)
    #   - Object ở góc? ✓  Object ở viền? ✓  Object ở giữa? ✓
    if aug_level == 'light':
        return transforms.Compose([
            agc_jit,                             # << AGC trước crop (mean toàn ảnh)
            transforms.RandomResizedCrop(        # << một transform thay thế Resize+Crop
                img_size,
                scale=(0.6, 1.0),               # crop 60-100% diện tích
                ratio=(3/4, 4/3),               # aspect ratio hợp lý
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.ColorJitter(0.15, 0.15, 0.15, 0.03),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    # ── Medium (Phase 2 & SWA) ────────────────────────────────────────────
    if aug_level == 'medium':
        return transforms.Compose([
            agc_jit,
            transforms.RandomResizedCrop(
                img_size,
                scale=(0.5, 1.0),               # crop rộng hơn → scale diversity
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.05),
            transforms.ColorJitter(0.25, 0.25, 0.25, 0.04),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.1, scale=(0.02, 0.08)),
        ])

    # ── Heavy (Phase 3 - Full fine-tune) ─────────────────────────────────
    # Thứ tự cố ý:
    #   1. AGC jitter TRƯỚC RandomResizedCrop → đo mean trên toàn ảnh gốc
    #   2. RandomResizedCrop → vị trí + scale ngẫu nhiên, không méo object
    #   3. Shadow / HardLight → inject residual noise CỤC BỘ sau khi đã crop
    #   4. ColorJitter mạnh → màu sắc / contrast
    #   5. Geometric aug → xoay, perspective
    if aug_level == 'heavy':
        return transforms.Compose([
            agc_jit,                             # Step 1: normalize toàn ảnh
            transforms.RandomResizedCrop(        # Step 2: vị trí & scale ngẫu nhiên
                img_size,
                scale=(0.4, 1.0),               # crop 40-100% → scale diversity mạnh nhất
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.1),
            RandomShadowInjection(p=0.45, dark_lo=0.3,  dark_hi=0.65),   # Step 3a
            RandomHardLight(p=0.30,      intensity_lo=1.4, intensity_hi=2.5),  # Step 3b
            transforms.ColorJitter(brightness=0.5, contrast=0.5,         # Step 4
                                   saturation=0.4, hue=0.08),
            transforms.RandomGrayscale(p=0.05),
            transforms.RandomRotation(20),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
            transforms.RandAugment(num_ops=2, magnitude=7),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
        ])

    # ── Lighting (Phase 5 - Domain Adaptation) ────────────────────────────
    # Giảm scale range (0.5-1.0 thay vì 0.4) để crop không quá nhỏ,
    # đảm bảo object vẫn đủ rõ sau khi inject shadow/hardlight.
    if aug_level == 'lighting':
        return transforms.Compose([
            agc_jit,
            transforms.RandomResizedCrop(
                img_size,
                scale=(0.5, 1.0),
                ratio=(3/4, 4/3),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(0.5),
            RandomWhiteBackgroundJitter(p=0.40, threshold=242, value_lo=235, value_hi=255, noise=4),
            RandomShadowInjection(p=0.55, dark_lo=0.25, dark_hi=0.70),
            RandomHardLight(p=0.45,      intensity_lo=1.5, intensity_hi=3.0),
            transforms.ColorJitter(brightness=0.6, contrast=0.55,
                                   saturation=0.5, hue=0.10),
            transforms.RandomGrayscale(p=0.08),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
        ])

    return get_transforms(img_size, 'heavy', config)   # fallback


def make_weighted_sampler(samples, power=0.7):
    counts = np.zeros(NUM_CLASSES, dtype=np.float32)
    for _, lbl in samples:
        counts[lbl] += 1
    wpc = 1.0 / np.power(np.maximum(counts, 1), power)
    sw  = torch.tensor([wpc[lbl] for _, lbl in samples], dtype=torch.float32)
    return WeightedRandomSampler(sw, len(sw), replacement=True)


def make_dataloaders(trn_samples, val_samples, img_size, batch_size,
                     aug_level, num_workers, device_type,
                     old_trn_dl=None, old_val_dl=None,
                     config=None):
    if old_trn_dl is not None: del old_trn_dl
    if old_val_dl is not None: del old_val_dl
    gc.collect()

    cfg    = config or CONFIG
    trn_tf = get_transforms(img_size, aug_level, cfg)
    val_tf = get_transforms(img_size, 'val',     cfg)   # << AGC deterministic

    trn_ds  = WasteDataset(trn_samples, trn_tf)
    val_ds  = WasteDataset(val_samples, val_tf)
    sampler = make_weighted_sampler(trn_samples, cfg.get('sampler_power', 0.7))
    pin     = (device_type == 'cuda' and num_workers > 0)

    trn_kw = dict(batch_size=batch_size, sampler=sampler,
                  num_workers=num_workers, pin_memory=pin, drop_last=True)
    val_bs = batch_size if img_size >= 384 else batch_size * 2
    val_kw = dict(batch_size=val_bs, shuffle=False,
                  num_workers=num_workers, pin_memory=pin)

    if num_workers > 0:
        trn_kw.update(persistent_workers=False, prefetch_factor=2)
        val_kw.update(persistent_workers=False, prefetch_factor=2)

    trn_dl = DataLoader(trn_ds, **trn_kw)
    val_dl = DataLoader(val_ds, **val_kw)
    print('Dataloaders ready.')
    return trn_dl, val_dl


# ── Quick sanity check ────────────────────────────────────────────────────────
def _demo_agc():
    agc = AdaptiveGammaCorrection(target=128)
    scenarios = [
        ('Đêm tối (mean=30)',      30),
        ('Trong nhà (mean=80)',    80),
        ('Lý tưởng (mean=128)',   128),
        ('Nắng buổi chiều (mean=180)', 180),
        ('Nắng trưa (mean=220)',  220),
    ]
    print('\n  AGC Gamma theo độ sáng thực tế:')
    print(f'  {"Scenario":<35} {"Mean V":>8} {"Gamma":>8} {"Action":>20}')
    print('  ' + '-'*75)
    for label, mean_v in scenarios:
        g = agc._compute_gamma(float(mean_v))
        action = 'KÉOSÁNG' if g < 0.95 else ('DÌM' if g > 1.05 else 'PASS-THROUGH')
        print(f'  {label:<35} {mean_v:>8} {g:>8.3f} {action:>20}')
    print()

_demo_agc()
print('Dataset & Augmentation module ready (v5 — Adaptive Gamma).')
print('  Transforms: AdaptiveGammaCorrection | RandomAdaptiveGammaJitter | '
      'RandomWhiteBackgroundJitter | RandomShadowInjection | RandomHardLight')

# ============================================================


  AGC Gamma theo độ sáng thực tế:
  Scenario                              Mean V    Gamma               Action
  ---------------------------------------------------------------------------
  Đêm tối (mean=30)                         30    0.400              KÉOSÁNG
  Trong nhà (mean=80)                       80    0.595              KÉOSÁNG
  Lý tưởng (mean=128)                      128    1.000         PASS-THROUGH
  Nắng buổi chiều (mean=180)               180    1.979                  DÌM
  Nắng trưa (mean=220)                     220    3.000                  DÌM

Dataset & Augmentation module ready (v5 — Adaptive Gamma).
  Transforms: AdaptiveGammaCorrection | RandomAdaptiveGammaJitter | RandomWhiteBackgroundJitter | RandomShadowInjection | RandomHardLight


In [4]:
# ============================================================
# CELL 4 - MODEL, LOSS, UTILITIES  (v3: + KnowledgeDistillationLoss)
# ============================================================

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), (1,1)
        ).pow(1.0 / self.p)


class WasteDetector(nn.Module):
    def __init__(self, num_classes, pretrained=True, dropout=0.4):
        super().__init__()
        weights       = MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        base          = mobilenet_v3_large(weights=weights)
        self.features = base.features
        self.gem_pool = GeM(p=3)
        in_f          = base.classifier[0].in_features  # 960

        self.classifier = nn.Sequential(
            nn.Linear(in_f, 512),
            nn.BatchNorm1d(512),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        self.objectness = nn.Sequential(
            nn.Linear(in_f, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        feat   = self.features(x)
        feat   = self.gem_pool(feat).flatten(1)
        logits = self.classifier(feat)
        obj    = self.objectness(feat)
        return logits, obj


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, targets):
        C = logits.size(1)
        if self.label_smoothing > 0:
            smooth   = self.label_smoothing / C
            one_hot  = torch.zeros_like(logits).scatter(1, targets.unsqueeze(1).long(), 1)
            one_hot  = one_hot * (1 - self.label_smoothing) + smooth
            log_prob = F.log_softmax(logits, dim=1)
            ce       = -(one_hot * log_prob).sum(1)
        else:
            ce = F.cross_entropy(logits, targets, reduction='none')
        pt      = torch.exp(-ce.clamp(min=0.0, max=100.0))
        focal_w = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets.view(-1).long())
            focal_w = focal_w * alpha_t
        return (focal_w * ce).mean()


# ══════════════════════════════════════════════════════════════════════════════
#  [v3] KNOWLEDGE DISTILLATION LOSS
# ══════════════════════════════════════════════════════════════════════════════
#
#  Lý thuyết (Hinton et al., 2015 "Distilling the Knowledge in a Neural Network"):
#    Student học từ "soft targets" của teacher thay vì chỉ học từ hard labels.
#    Soft targets chứa "dark knowledge" — ví dụ teacher tin rằng ảnh Battery có
#    5% khả năng là Metal (hình dáng tương tự). Hard label (Battery=1, rest=0)
#    bỏ mất thông tin quan trọng này.
#
#  Công thức:
#    L_KD = KL(softmax(student/T) || softmax(teacher/T)) × T²
#    T > 1 làm mềm distribution → gradient KL rõ ràng hơn ở mọi class
#    nhân T² để scale loss về cùng magnitude với cross-entropy
#
#  Tại sao hữu ích cho Phase 6?
#    - Teacher (Phase 5) đã học tốt 7 classes trên old data
#    - Student (Phase 6) train trên old+real → có nguy cơ quên old data
#    - KD loss buộc student output ≈ teacher output → giữ old knowledge
#    - Hiệu quả hơn chỉ dùng LR nhỏ vì: KD trực tiếp penalize deviation
#      thay vì chỉ slow down learning rate
#
#  total_loss = (1 - kd_weight) × focal_loss + kd_weight × kd_loss
#             = 0.7 × focal + 0.3 × KD   (với kd_weight=0.3)
# ══════════════════════════════════════════════════════════════════════════════

class KnowledgeDistillationLoss(nn.Module):
    """
    KL divergence giữa student soft targets và teacher soft targets.

    Args:
        temperature : T > 1 làm mềm soft targets. T=4 là giá trị kinh nghiệm tốt.
                      T=1 → hard KD (gần giống CE). T=10+ → quá mềm, ít hiệu quả.
    """
    def __init__(self, temperature: float = 4.0):
        super().__init__()
        self.T = temperature

    def forward(self, student_logits: torch.Tensor,
                teacher_logits: torch.Tensor) -> torch.Tensor:
        """
        Args:
            student_logits : raw logits từ student model  [B, C]
            teacher_logits : raw logits từ teacher model  [B, C] (detached, no grad)
        Returns:
            scalar KD loss
        """
        # log-softmax cho student (numerically stable)
        s = F.log_softmax(student_logits / self.T, dim=1)
        # softmax cho teacher (detached — không backprop vào teacher)
        t = F.softmax(teacher_logits.detach() / self.T, dim=1)
        # KL divergence × T² để rescale
        return F.kl_div(s, t, reduction='batchmean') * (self.T ** 2)

    def __repr__(self):
        return f'KnowledgeDistillationLoss(T={self.T})'


def compute_class_alpha(samples, power=0.7):
    counts = np.zeros(NUM_CLASSES, dtype=np.float32)
    for _, lbl in samples:
        counts[lbl] += 1
    alpha = 1.0 / np.power(np.maximum(counts, 1), power)
    return (alpha / alpha.sum() * NUM_CLASSES).tolist()


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam  = max(np.random.beta(alpha, alpha), 1.0 - np.random.beta(alpha, alpha))
    ridx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[ridx], y, y[ridx], lam


def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0: return x, y, y, 1.0
    lam  = max(np.random.beta(alpha, alpha), 1.0 - np.random.beta(alpha, alpha))
    ridx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cr   = math.sqrt(1.0 - lam)
    cw, ch = int(W*cr), int(H*cr)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1,x2 = max(cx-cw//2,0), min(cx+cw//2,W)
    y1,y2 = max(cy-ch//2,0), min(cy+ch//2,H)
    mx = x.clone(); mx[:,:,y1:y2,x1:x2] = x[ridx,:,y1:y2,x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return mx, y, y[ridx], lam


def mixed_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)


class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, epoch=0):
        d = min(self.decay, (1 + epoch) / (10 + epoch))
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.data.mul_(d).add_(mp.data, alpha=1-d)
        for eb, mb in zip(self.ema.buffers(), model.buffers()):
            eb.data.copy_(mb.data)


def get_backbone_groups(model):
    f = model.features
    return [list(f[0:4].parameters()), list(f[4:7].parameters()),
            list(f[7:13].parameters()), list(f[13:].parameters())]

def freeze_backbone(model):
    for p in model.features.parameters():
        p.requires_grad = False

def unfreeze_groups(model, idxs):
    for g in idxs:
        for p in get_backbone_groups(model)[g]:
            p.requires_grad = True

def make_param_groups(model, lr_head, lr_bb):
    pgs = []
    for gi, params in enumerate(get_backbone_groups(model)):
        trainable = [p for p in params if p.requires_grad]
        if trainable:
            scale = max(2**(gi-3), 0.125)
            pgs.append({'params': trainable, 'lr': lr_bb * scale, 'name': f'bb_g{gi}'})
    head_p = (list(model.classifier.parameters()) +
              list(model.objectness.parameters()) +
              list(model.gem_pool.parameters()))
    pgs.append({'params': [p for p in head_p if p.requires_grad],
                'lr': lr_head, 'name': 'head'})
    return pgs

print('Model & Loss module ready (v3: + KnowledgeDistillationLoss).')


Model & Loss module ready (v3: + KnowledgeDistillationLoss).


In [5]:
# ============================================================
# CELL 5 - TRAIN / VALIDATE LOOPS  (v3: KD-aware + tight clip)
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device,
                    mixup=0.0, cutmix=0.0, scheduler=None,
                    scaler=None, epoch=0, clip_grad=5.0):
    """Standard training loop (Phase 1-5). clip_grad default = 5.0 cho compat."""
    model.train()
    total_loss = correct = total = 0
    use_amp = (device.type == 'cuda' and scaler is not None and scaler.is_enabled())

    pbar = tqdm(loader, desc='  Train', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        min_label = labels.min().item(); max_label = labels.max().item()
        if min_label < 0 or max_label >= NUM_CLASSES:
            raise ValueError(f'Invalid labels: min={min_label}, max={max_label}')

        _nb = (device.type == 'cuda')
        imgs   = imgs.to(device, non_blocking=_nb)
        labels = labels.to(device, non_blocking=_nb)

        use_cut = cutmix > 0 and np.random.rand() < 0.5
        use_mix = mixup  > 0 and not use_cut
        if use_cut:   imgs, la, lb, lam = cutmix_data(imgs, labels, cutmix)
        elif use_mix: imgs, la, lb, lam = mixup_data(imgs, labels, mixup)
        else:         la, lb, lam = labels, labels, 1.0

        _amp_ctx = (torch.autocast(device_type='cuda', enabled=True)
                    if use_amp else contextlib.nullcontext())
        with _amp_ctx:
            logits, _ = model(imgs)
            loss = (mixed_criterion(criterion, logits, la, lb, lam)
                    if lam < 1.0 else criterion(logits, labels))

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.detach().argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.3f}')
        del imgs, labels, logits, loss

    return total_loss / total, correct / total


# ══════════════════════════════════════════════════════════════════════════════
#  [v3] TRAIN ONE EPOCH WITH KNOWLEDGE DISTILLATION
# ══════════════════════════════════════════════════════════════════════════════
def train_one_epoch_kd(model, teacher, loader, optimizer,
                       criterion, kd_criterion, device,
                       kd_weight=0.3, mixup=0.0, scaler=None,
                       epoch=0, clip_grad=1.0):
    """
    Training loop với Knowledge Distillation.

    total_loss = (1 - kd_weight) × focal_loss + kd_weight × kd_loss

    Notes:
    - teacher luôn ở eval mode, không cần gradient
    - KD loss dùng raw logits (không mixup) để teacher/student có cùng input
    - Mixup chỉ áp dụng cho focal_loss (không áp dụng cho KD)
      Lý do: mixup(img_A, img_B) → student thấy interpolation, nhưng teacher
      cần đánh giá trên ảnh gốc để soft targets có nghĩa
    - clip_grad = 1.0 cho Phase 6 (v3 fix từ 5.0)
    """
    model.train()
    teacher.eval()   # teacher luôn eval, không train
    total_loss = total_focal = total_kd = correct = total = 0
    use_amp = (device.type == 'cuda' and scaler is not None and scaler.is_enabled())

    pbar = tqdm(loader, desc='  Train KD', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        min_label = labels.min().item(); max_label = labels.max().item()
        if min_label < 0 or max_label >= NUM_CLASSES:
            raise ValueError(f'Invalid labels: min={min_label}, max={max_label}')

        _nb = (device.type == 'cuda')
        imgs_orig = imgs.to(device, non_blocking=_nb)   # ảnh gốc cho teacher
        labels    = labels.to(device, non_blocking=_nb)

        # Mixup chỉ cho focal loss — không áp dụng vào ảnh teacher
        use_mix = mixup > 0 and np.random.rand() < 0.5
        if use_mix:
            imgs_mix, la, lb, lam = mixup_data(imgs_orig, labels, mixup)
        else:
            imgs_mix, la, lb, lam = imgs_orig, labels, labels, 1.0

        _amp_ctx = (torch.autocast(device_type='cuda', enabled=True)
                    if use_amp else contextlib.nullcontext())
        with _amp_ctx:
            # Student forward — trên ảnh mixed (hoặc gốc nếu không mixup)
            student_logits, _ = model(imgs_mix)

            # Focal loss (với mixup nếu có)
            focal = (mixed_criterion(criterion, student_logits, la, lb, lam)
                     if lam < 1.0 else criterion(student_logits, labels))

            # Teacher forward — luôn trên ảnh GỐC (không mixup)
            with torch.no_grad():
                teacher_logits, _ = teacher(imgs_orig)

            # Nếu mixup: student logits cũng từ ảnh gốc để KD match với teacher
            # → dùng student logits trên imgs_orig nếu đang mixup
            if use_mix:
                with _amp_ctx:
                    student_logits_orig, _ = model(imgs_orig)
                kd_loss = kd_criterion(student_logits_orig, teacher_logits)
            else:
                kd_loss = kd_criterion(student_logits, teacher_logits)

            loss = (1.0 - kd_weight) * focal + kd_weight * kd_loss

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)   # [v3] 1.0
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)   # [v3] 1.0
            optimizer.step()

        total_loss  += loss.item()  * imgs_orig.size(0)
        total_focal += focal.item() * imgs_orig.size(0)
        total_kd    += kd_loss.item() * imgs_orig.size(0)
        preds_cls    = student_logits.detach().argmax(1)
        correct     += (preds_cls == labels).sum().item()
        total       += labels.size(0)
        pbar.set_postfix(
            L=f'{loss.item():.3f}',
            focal=f'{focal.item():.3f}',
            kd=f'{kd_loss.item():.3f}'
        )
        del imgs_orig, imgs_mix, labels, student_logits, teacher_logits, loss, focal, kd_loss

    return (total_loss/total, total_focal/total, total_kd/total, correct/total)


@torch.no_grad()
def validate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss = correct = total = 0
    cls_correct = defaultdict(int); cls_total = defaultdict(int)
    use_amp     = (device.type == 'cuda')
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        min_label = labels.min().item(); max_label = labels.max().item()
        if min_label < 0 or max_label >= NUM_CLASSES:
            raise ValueError(f'Invalid labels in val: min={min_label}, max={max_label}')

        _nb = (device.type == 'cuda')
        imgs   = imgs.to(device, non_blocking=_nb)
        labels = labels.to(device, non_blocking=_nb)

        _amp_ctx = (torch.autocast(device_type='cuda', enabled=True)
                    if use_amp else contextlib.nullcontext())
        with _amp_ctx:
            logits, _ = model(imgs)
            loss      = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds_cpu  = logits.detach().argmax(1).cpu()
        labels_cpu = labels.cpu()
        correct   += (preds_cpu == labels_cpu).sum().item()
        total     += labels_cpu.size(0)

        for p, l in zip(preds_cpu.numpy(), labels_cpu.numpy()):
            cls_total[l] += 1; cls_correct[l] += int(p == l)

        if return_preds:
            all_preds.extend(preds_cpu.numpy())
            all_labels.extend(labels_cpu.numpy())
        del imgs, labels, logits, loss

    per_cls = {IDX2CLASS[k]: cls_correct[k]/max(cls_total[k],1)
               for k in sorted(cls_total)}
    if return_preds:
        return total_loss/total, correct/total, per_cls, all_preds, all_labels
    return total_loss/total, correct/total, per_cls


def _track(h, tl, ta, vl, va, ea, phase, lr, focal=None, kd=None, real_acc=None):
    h['train_loss'].append(tl); h['train_acc'].append(ta)
    h['val_loss'].append(vl);   h['val_acc'].append(va)
    h['ema_val_acc'].append(ea); h['phase'].append(phase); h['lr'].append(lr)
    if focal is not None: h.setdefault('focal_loss', []).append(focal)
    if kd    is not None: h.setdefault('kd_loss',    []).append(kd)
    if real_acc is not None: h.setdefault('real_val_acc', []).append(real_acc)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    elif hasattr(torch.mps, 'empty_cache'):
        torch.mps.empty_cache()


def gpu_mem_str():
    if not torch.cuda.is_available(): return ''
    used  = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f' [VRAM {used:.1f}/{total:.1f}GB]'


print('Training loops ready (v3: train_one_epoch_kd + tight clip_grad).')


Training loops ready (v3: train_one_epoch_kd + tight clip_grad).


In [6]:
# ============================================================
# CELL 6 - VISUALIZATION  (v3: + real_val_acc curve)
# ============================================================

def plot_history(history, out_dir):
    """
    v3: plot thêm real_val_acc và focal/kd loss breakdown nếu có.
    """
    has_kd      = 'kd_loss' in history and len(history['kd_loss']) > 0
    has_realval = 'real_val_acc' in history and len(history['real_val_acc']) > 0
    n_cols = 3 + (1 if has_kd else 0) + (1 if has_realval else 0)

    fig, axes = plt.subplots(1, n_cols, figsize=(6*n_cols, 5))
    ep = range(1, len(history['train_loss']) + 1)
    ax_idx = 0

    # Loss
    axes[ax_idx].plot(ep, history['train_loss'], label='Train')
    axes[ax_idx].plot(ep, history['val_loss'],   label='Val')
    axes[ax_idx].set_title('Total Loss'); axes[ax_idx].legend(); axes[ax_idx].grid(True)
    ax_idx += 1

    # Accuracy
    axes[ax_idx].plot(ep, history['train_acc'],   label='Train')
    axes[ax_idx].plot(ep, history['val_acc'],     label='Val')
    if any(v > 0 for v in history['ema_val_acc']):
        axes[ax_idx].plot(ep, history['ema_val_acc'], label='EMA (old)', ls='--', alpha=0.7)
    axes[ax_idx].set_title('Accuracy'); axes[ax_idx].legend(); axes[ax_idx].grid(True)
    ax_idx += 1

    # LR
    axes[ax_idx].plot(ep, history['lr'], color='tab:orange')
    axes[ax_idx].set_title('Learning Rate')
    axes[ax_idx].set_yscale('log'); axes[ax_idx].grid(True)
    ax_idx += 1

    # [v3] KD vs Focal loss breakdown
    if has_kd:
        axes[ax_idx].plot(ep, history.get('focal_loss', [0]*len(ep)), label='Focal', color='tab:blue')
        axes[ax_idx].plot(ep, history['kd_loss'],                     label='KD',    color='tab:red')
        axes[ax_idx].set_title('Focal vs KD Loss'); axes[ax_idx].legend(); axes[ax_idx].grid(True)
        ax_idx += 1

    # [v3] Real val accuracy — domain adaptation monitor
    if has_realval:
        axes[ax_idx].plot(ep, history['real_val_acc'], color='tab:green',
                          marker='o', markersize=3, label='Real val acc')
        axes[ax_idx].axhline(history['real_val_acc'][0], color='gray', ls=':', alpha=0.5,
                             label=f'Baseline {history["real_val_acc"][0]:.3f}')
        axes[ax_idx].set_title('Real-World Val Accuracy')
        axes[ax_idx].set_ylabel('Accuracy'); axes[ax_idx].legend(); axes[ax_idx].grid(True)
        axes[ax_idx].set_ylim(0, 1)
        ax_idx += 1

    # Phase boundaries
    for i in range(1, len(history['phase'])):
        if history['phase'][i] != history['phase'][i-1]:
            for ax in axes:
                ax.axvline(i+1, color='gray', ls=':', alpha=0.5)

    plt.tight_layout()
    path = os.path.join(out_dir, 'training_history_p6.png')
    plt.savefig(path, dpi=120); plt.close(); plt.clf()
    print(f'  >> Plot: {path}')


def plot_confusion_matrix(preds, labels, out_dir, suffix=''):
    label_ids = list(range(NUM_CLASSES))
    cm = confusion_matrix(labels, preds, labels=label_ids)
    row_sum = cm.sum(1, keepdims=True)
    cm_norm = np.zeros_like(cm, dtype=np.float32)
    np.divide(cm.astype(np.float32), row_sum, out=cm_norm, where=row_sum != 0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    axes[0].set_title('Confusion (counts)'); axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[1].set_title('Confusion (normalized)'); axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    path = os.path.join(out_dir, f'confusion_matrix{suffix}.png')
    plt.savefig(path, dpi=120); plt.close(); plt.clf()
    print(f'  >> Confusion: {path}')
    print(classification_report(labels, preds, labels=label_ids,
                                 target_names=CLASSES, zero_division=0))

print('Visualization ready (v3: + real_val curve + KD breakdown).')


Visualization ready (v3: + real_val curve + KD breakdown).


In [7]:
# ============================================================
# CELL 7 - DATA PREPARATION  (v3: Real split + Per-Class Oversample)
# ============================================================
#
#  Chiến lược v3 sau khi real data đã phong phú hơn và đã remove background:
#    - OLD Dataset: 100% → split 80/20 → train_old / val_old
#    - REAL Dataset: stratified split → train_real / real_monitor
#    - train_real được per-class adaptive oversample để học domain camera/nền trắng
#    - real_monitor chỉ LOG real accuracy, không dùng early stopping và không backprop
#
#  Vì nền thực tế là trắng, ảnh remove-background là hợp lý nếu inference cũng thấy nền trắng.
#  Pipeline vẫn jitter nhẹ vùng nền trắng để tránh model học thuộc viền cắt nền.
#
# ĐỔI 2 ĐƯỜNG DẪN NÀY TRƯỚC KHI CHẠY:
DATA_DIR_OLD  = '../DataSet/Data'          # Dataset gốc v1
DATA_DIR_REAL = '../DataSet/DataRealTest'  # Ảnh thực tế đã remove background/nền trắng
# ============================================================

import random as _random
from collections import defaultdict as _defaultdict

OLD_DATASET_RATIO      = CONFIG.get('p6_old_ratio', 1.0)
REAL_OVERSAMPLE_FACTOR = CONFIG.get('p6_real_oversample_factor', 8)
REAL_MIN_TARGET        = CONFIG.get('p6_real_min_target', 1500)
REAL_TARGET_CAP        = CONFIG.get('p6_real_target_cap', 3500)
REAL_VAL_PCT           = CONFIG.get('p6_real_val_pct', 0.15)
REAL_VAL_MIN_PER_CLASS = CONFIG.get('p6_real_val_min_per_class', 5)
VAL_RATIO              = CONFIG.get('p6_val_ratio', 0.20)
VAL_SOURCE             = 'old'


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_samples_from_dir(data_dir, desc=''):
    data_path    = _resolve_data_root(data_dir)
    print(f'[INFO] {desc} root: {data_path}')

    folder_map       = {f.name: f for f in data_path.iterdir() if f.is_dir()}
    canonical_to_dir = {}
    for fn, fp in folder_map.items():
        cn = _canonical_from_folder(fn)
        if cn and cn not in canonical_to_dir:
            canonical_to_dir[cn] = fp
    for cls_name in CLASSES:
        exact = data_path / cls_name
        if exact.exists() and exact.is_dir():
            canonical_to_dir[cls_name] = exact

    missing = [c for c in CLASSES if c not in canonical_to_dir]
    if missing:
        print(f'[WARN] Missing class folders: {missing}')

    samples = []; counts = {}
    for cls_name in CLASSES:
        if cls_name not in canonical_to_dir:
            counts[cls_name] = 0; continue
        cls_dir = canonical_to_dir[cls_name]; cls_idx = CLASS2IDX[cls_name]; cnt = 0
        for ext in ('*.jpg','*.jpeg','*.png','*.bmp','*.webp',
                    '*.JPG','*.JPEG','*.PNG','*.BMP','*.WEBP'):
            for p in cls_dir.glob(ext):
                samples.append((str(p), cls_idx)); cnt += 1
        counts[cls_name] = cnt
    return samples, counts


def stratified_subsample(samples, ratio, seed=42):
    if ratio >= 1.0:
        return samples[:]
    labels = [lbl for _, lbl in samples]
    sss    = StratifiedShuffleSplit(n_splits=1, test_size=(1.0-ratio), random_state=seed)
    idx_keep, _ = next(sss.split(samples, labels))
    return [samples[i] for i in sorted(idx_keep)]


def train_real_monitor_split(samples, val_ratio=0.15, min_val_per_class=5, seed=42):
    """Tách real monitor theo từng class để metric real không quá nhiễu ở class ít ảnh."""
    rng = np.random.RandomState(seed)
    by_cls = _defaultdict(list)
    for sample in samples:
        by_cls[sample[1]].append(sample)

    train, monitor = [], []
    for lbl in range(NUM_CLASSES):
        cls_samples = by_cls.get(lbl, [])
        if not cls_samples:
            continue
        idxs = rng.permutation(len(cls_samples))
        desired = int(round(len(cls_samples) * val_ratio))
        if len(cls_samples) >= min_val_per_class * 2:
            val_n = max(min_val_per_class, desired)
        else:
            val_n = max(1, desired)
        val_n = min(val_n, len(cls_samples) - 1)
        monitor.extend(cls_samples[i] for i in idxs[:val_n])
        train.extend(cls_samples[i] for i in idxs[val_n:])

    rng.shuffle(train)
    rng.shuffle(monitor)
    return train, monitor


def adaptive_class_oversample(real_samples, factor=8, seed=42, min_target=1500, target_cap=3500):
    """
    Per-class adaptive oversample — v3 dùng 100% real data.

    Thuật toán:
      1. max_count = số ảnh của class nhiều nhất trong real
      2. target_per_class = max_count × factor
      3. Mỗi class repeat k = ceil(target/n) lần, trim về target
      → Class ít ảnh → multiplier cao hơn → cân bằng real distribution
      → Augmentation random tại load time → mỗi repeat = biến thể khác nhau

    Tại sao quan trọng?
      v2: ×30 đồng đều → class 3 ảnh: 90 virtual; class 30 ảnh: 900 virtual
          → model thấy class 30 ảnh nhiều hơn 10× → mất cân bằng real
      v3: target đồng đều → mọi class đều có target virtual samples
          → model học đều nhau cho tất cả classes real
    """
    rng = np.random.RandomState(seed)
    cls_samples = _defaultdict(list)
    for path, lbl in real_samples:
        cls_samples[lbl].append((path, lbl))

    if not cls_samples:
        return []

    class_counts = {lbl: len(s) for lbl, s in cls_samples.items()}
    max_count    = max(class_counts.values())
    raw_target   = max(max_count * factor, min_target)
    target       = min(raw_target, target_cap) if target_cap else raw_target

    print(f'  Adaptive oversample: max_class_count={max_count}  raw_target={raw_target:,}  target_per_class={target:,}')

    result = []
    for lbl in range(NUM_CLASSES):
        smpls = cls_samples.get(lbl, [])
        if not smpls:
            continue
        n  = len(smpls)
        k  = (target + n - 1) // n   # ceiling division
        rp = smpls * k
        idxs = rng.permutation(len(rp))
        trimmed = [rp[i] for i in idxs[:target]]
        result.extend(trimmed)
        print(f'    {IDX2CLASS[lbl]:<20}: {n:>3} ảnh × {k}× → {len(trimmed):,} virtual')

    return result


# ══════════════════════════════════════════════════════════════
# BƯỚC 1 — Load OLD dataset nếu có
# ══════════════════════════════════════════════════════════════
print('=' * 65)
print(f'  BƯỚC 1: Load OLD dataset  (dùng {OLD_DATASET_RATIO:.0%})')
print('=' * 65)

try:
    all_old_samples, _ = load_samples_from_dir(DATA_DIR_OLD, 'OLD')
except FileNotFoundError:
    all_old_samples = []
    trn_old = []
    val_samples = []
    VAL_SOURCE = 'real_monitor'
    print(f'  [WARN] Không tìm thấy OLD dataset: {DATA_DIR_OLD}')
    print('         Notebook sẽ train bằng real data + KD từ checkpoint V2.')
    print('         Để tối ưu hơn, đặt old dataset tại Model/DataSet/Data hoặc chỉnh DATA_DIR_OLD.')
else:
    old_used = stratified_subsample(
        all_old_samples, ratio=OLD_DATASET_RATIO, seed=CONFIG['seed'])
    trn_old, val_samples = train_val_split_stratified(
        old_used, val_ratio=VAL_RATIO, seed=CONFIG['seed'])

    print(f'  OLD tổng cộng : {len(all_old_samples):,}')
    print(f'  Sau subsample : {len(old_used):,}  ({OLD_DATASET_RATIO:.0%})')
    print(f'  → train_old   : {len(trn_old):,}')
    print(f'  → val_old     : {len(val_samples):,}  (dùng cho early stop)')


# ══════════════════════════════════════════════════════════════
# BƯỚC 2 — Load REAL dataset → split train/monitor
# ══════════════════════════════════════════════════════════════
print('\n' + '=' * 65)
print(f'  BƯỚC 2: Load REAL dataset — train + monitor split ({REAL_VAL_PCT:.0%} monitor)')
print('=' * 65)

real_samples, real_counts = load_samples_from_dir(DATA_DIR_REAL, 'REAL')

if not real_samples:
    raise RuntimeError(
        f"Không tìm thấy ảnh trong: {DATA_DIR_REAL}\n"
        f"Kiểm tra lại DATA_DIR_REAL."
    )

print(f'\n  Real tổng cộng: {len(real_samples)}')
print(f'  Phân bổ per class:')
for cls in CLASSES:
    cnt = real_counts.get(cls, 0)
    bar = '█' * min(cnt, 30) + '░' * max(0, 30 - cnt)
    print(f'    {cls:<20}: {cnt:>3}  {bar}')

if REAL_VAL_PCT > 0:
    train_real_samples, real_monitor_samples = train_real_monitor_split(
        real_samples, val_ratio=REAL_VAL_PCT,
        min_val_per_class=REAL_VAL_MIN_PER_CLASS, seed=CONFIG['seed'])
else:
    train_real_samples = real_samples[:]
    real_monitor_samples = []

print(f'\n  → train_real   : {len(train_real_samples):,} ảnh')
print(f'  → real_monitor : {len(real_monitor_samples):,} ảnh (LOG only, không train; min/class={REAL_VAL_MIN_PER_CLASS})')

if VAL_SOURCE == 'real_monitor':
    if not real_monitor_samples:
        raise RuntimeError('Không có OLD dataset và real_monitor rỗng. Tăng p6_real_val_pct > 0.')
    val_samples = real_monitor_samples
    print('  [INFO] Early stopping/validation sẽ dùng real_monitor vì OLD dataset không có.')

train_real_virtual = adaptive_class_oversample(
    train_real_samples, factor=REAL_OVERSAMPLE_FACTOR, seed=CONFIG['seed'],
    min_target=REAL_MIN_TARGET, target_cap=REAL_TARGET_CAP)

print(f'\n  train_real_virtual: {len(train_real_virtual):,} virtual samples')
print(f'  oversample policy : factor={REAL_OVERSAMPLE_FACTOR}x, min_target={REAL_MIN_TARGET:,}, cap={REAL_TARGET_CAP:,}')


# ══════════════════════════════════════════════════════════════
# BƯỚC 3 — Ghép TRAIN = train_old + train_real_virtual
# ══════════════════════════════════════════════════════════════
print('\n' + '=' * 65)
print('  BƯỚC 3: Ghép TRAIN')
print('=' * 65)

trn_samples = trn_old + train_real_virtual
_random.seed(CONFIG['seed'])
_random.shuffle(trn_samples)

real_pct = len(train_real_virtual) / len(trn_samples) * 100
print(f'\n  train_old            : {len(trn_old):,}')
print(f'  train_real_source    : {len(train_real_samples):,}  (real train, no monitor overlap)')
print(f'  train_real_virtual   : {len(train_real_virtual):,}  (per-class adaptive)')
print(f'  ─────────────────────────────────────────────')
print(f'  TRAIN tổng cộng      : {len(trn_samples):,}')
print(f'  Real chiếm           : {real_pct:.1f}%')
print(f'  VAL ({VAL_SOURCE})       : {len(val_samples):,}  (early stop)')
print(f'  MONITOR (real holdout): {len(real_monitor_samples)}  (LOG only)')

# Bảng phân phối
print(f'\n  Phân phối class trong TRAIN:')
trn_counts = {}
for _, lbl in trn_samples:
    c = IDX2CLASS[lbl]; trn_counts[c] = trn_counts.get(c, 0) + 1
real_virtual_by_cls = {}
for _, lbl in train_real_virtual:
    c = IDX2CLASS[lbl]; real_virtual_by_cls[c] = real_virtual_by_cls.get(c, 0) + 1
val_counts = {}
for _, lbl in val_samples:
    c = IDX2CLASS[lbl]; val_counts[c] = val_counts.get(c, 0) + 1
real_monitor_by_cls = {}
for _, lbl in real_monitor_samples:
    c = IDX2CLASS[lbl]; real_monitor_by_cls[c] = real_monitor_by_cls.get(c, 0) + 1

print(f'  {"Class":<20} {"Train total":>12} {"(real virtual)":>16} {"Val":>9} {"Real mon":>9}')
print('  ' + '─' * 74)
for cls in CLASSES:
    t  = trn_counts.get(cls, 0)
    rv = real_virtual_by_cls.get(cls, 0)
    v  = val_counts.get(cls, 0)
    rm = real_monitor_by_cls.get(cls, 0)
    flag = ' ← KHÔNG có real!' if real_counts.get(cls, 0) == 0 else ''
    print(f'  {cls:<20} {t:>12,} {rv:>16,} {v:>9} {rm:>9}{flag}')

missing_cls = [c for c in CLASSES if real_counts.get(c, 0) == 0]
if missing_cls:
    print(f'\n  [WARN] Classes chưa có real data: {missing_cls}')
    print('         Model sẽ không cải thiện real-world accuracy cho class này.')
    print('         → Thu thập thêm ảnh thực tế cho class này.')

print(f'\n[OK] Data preparation v3 done — validation source: {VAL_SOURCE}.')


  BƯỚC 1: Load OLD dataset  (dùng 100%)
[INFO] OLD root: ../DataSet/Data
  OLD tổng cộng : 19,012
  Sau subsample : 19,012  (100%)
  → train_old   : 15,209
  → val_old     : 3,803  (dùng cho early stop)

  BƯỚC 2: Load REAL dataset — train + monitor split (15% monitor)
[INFO] REAL root: ../DataSet/DataRealTest

  Real tổng cộng: 1124
  Phân bổ per class:
    Battery             : 401  ██████████████████████████████
    Biological          :  54  ██████████████████████████████
    General_Waste       : 144  ██████████████████████████████
    Glass               :  23  ███████████████████████░░░░░░░
    Metal               : 119  ██████████████████████████████
    Paper_Cardboard     : 201  ██████████████████████████████
    Plastic             : 182  ██████████████████████████████

  → train_real   : 954 ảnh
  → real_monitor : 170 ảnh (LOG only, không train; min/class=5)
  Adaptive oversample: max_class_count=341  raw_target=2,728  target_per_class=2,728
    Battery             : 341 ản

In [8]:
# ============================================================
# CELL 8 - PHASE 6: REAL-WORLD FINE-TUNING v3
# ============================================================
# Cải tiến so với v2:
#  1. Knowledge Distillation từ frozen Phase 5 teacher
#  2. clip_grad_norm_ = 1.0 (v2: 5.0)
#  3. EMA decay = 0.9995 (v2: 0.9998)
#  4. real_val monitoring — log domain adaptation progress
#  5. Per-class adaptive oversample data đã chuẩn bị ở Cell 7
# ============================================================

def phase6_finetune_v3(trn_samples, val_samples, real_monitor_samples,
                       config=CONFIG, base_ckpt=None, extra_epochs=None):
    """
    Phase 6 Fine-Tuning với Knowledge Distillation.

    Args:
        trn_samples      : train_old + train_real_virtual (từ Cell 7)
        val_samples      : val_old (ổn định, dùng early stop)
        real_monitor_samples : real holdout (LOG only, không dùng early stop)
        base_ckpt        : path đến Phase 5 checkpoint (teacher + student init)
    """
    device   = torch.device(config['device'])
    epochs   = extra_epochs or config.get('p6_epochs', 20)
    img_size = config.get('p6_img_size', 384)
    kd_w     = config.get('p6_kd_weight', 0.3)
    kd_T     = config.get('p6_kd_temperature', 4.0)
    clip_g   = config.get('p6_clip_grad', 1.0)
    selection_metric = config.get('p6_selection_metric', 'real')
    real_score_w = config.get('p6_real_score_weight', 0.80)

    print(f"\n{'='*70}")
    print(f"  PHASE 6 v3: REAL-WORLD FINE-TUNING + KNOWLEDGE DISTILLATION")
    print(f"  Device: {device}  |  Epochs: {epochs}  |  Batch: {config.get('p6_batch',8)}")
    print(f"  LR: head={config['p6_lr_head']:.0e}  backbone={config['p6_lr_backbone']:.0e}")
    print(f"  KD: T={kd_T}  weight={kd_w}  (focal_weight={1-kd_w})")
    print(f"  clip_grad={clip_g}  EMA_decay={config.get('p6_ema_decay', 0.9995)}")
    print(f"  selection={selection_metric}  real_weight={real_score_w}")
    print(f"{'='*70}\n")

    # ── Load checkpoint (teacher = student init) ──────────────────────────
    ckpt_to_load = base_ckpt or config.get('p6_base_ckpt', 'outputs_v2/best_model_v2.pth')
    out_dir      = config.get('output_dir', './outputs_v3')
    ckpt_p6      = os.path.join(out_dir, 'best_model_p6_v3.pth')
    os.makedirs(out_dir, exist_ok=True)

    # Student model
    model = WasteDetector(NUM_CLASSES, pretrained=False).to(device)

    if os.path.exists(ckpt_to_load):
        ck        = torch.load(ckpt_to_load, map_location=device, weights_only=False)
        state     = ck.get('model_state', ck.get('ema_state', ck))
        model.load_state_dict(state, strict=True)
        base_best = float(ck.get('val_acc', 0.0))
        src       = ck.get('source', 'unknown')
        print(f'  [OK] Student loaded: {ckpt_to_load}')
        print(f'       val_acc={base_best:.4f}  source={src}')
    else:
        raise FileNotFoundError(
            f"Checkpoint không tìm thấy: {ckpt_to_load}\n"
            f"Đặt CONFIG['p6_base_ckpt'] = path đến .pth của Phase 5."
        )

    # Teacher model — frozen copy của Phase 5
    # Teacher và student bắt đầu từ CÙNG weights (Phase 5 best model).
    # KD buộc student không đi xa khỏi teacher khi học real data.
    teacher = WasteDetector(NUM_CLASSES, pretrained=False).to(device)
    teacher.load_state_dict(state, strict=True)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    print(f'  [OK] Teacher (frozen Phase 5) ready.')
    print(f'       Teacher eval mode — no gradients.')

    # ── Dataloaders ────────────────────────────────────────────────────────
    batch_size = config.get('p6_batch', _B['p3'] * _SCALE)
    trn_dl, val_dl = make_dataloaders(
        trn_samples, val_samples, img_size, batch_size,
        config.get('p6_aug', 'lighting'),
        config['num_workers'], device.type, config=config
    )

    # real_val dataloader (nhỏ, batch không cần lớn)
    real_val_dl = None
    if real_monitor_samples:
        val_tf      = get_transforms(img_size, 'val', config)
        real_val_ds = WasteDataset(real_monitor_samples, val_tf)
        real_val_dl = DataLoader(
            real_val_ds,
            batch_size=min(batch_size, len(real_monitor_samples)),
            shuffle=False,
            num_workers=config['num_workers'],
            pin_memory=(device.type == 'cuda' and config['num_workers'] > 0),
        )
        print(f'  real_monitor: {len(real_monitor_samples)} ảnh (real holdout, LOG only)')

    # ── Optimizer & Scheduler ─────────────────────────────────────────────
    for p in model.parameters():
        p.requires_grad = True

    focal_alpha = compute_class_alpha(trn_samples, power=config.get('alpha_power', 0.7))
    criterion   = FocalLoss(
        config['focal_gamma'], focal_alpha,
        label_smoothing=config.get('p6_label_smooth', 0.02)
    ).to(device)
    kd_criterion = KnowledgeDistillationLoss(temperature=kd_T).to(device)

    param_groups = make_param_groups(
        model, lr_head=config['p6_lr_head'], lr_bb=config['p6_lr_backbone'])
    optimizer = optim.AdamW(param_groups, weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=2e-7)

    # [v3] EMA decay = 0.9995
    ema    = ModelEMA(model, decay=config.get('p6_ema_decay', 0.9995))
    scaler = (torch.amp.GradScaler(enabled=True) if device.type == 'cuda'
              else torch.amp.GradScaler('cpu', enabled=False))

    # ── Training state ─────────────────────────────────────────────────────
    best_score = -1.0
    best_acc   = 0.0
    best_real  = 0.0
    best_epoch = 0
    best_src   = 'init'
    best_w     = copy.deepcopy(model.state_dict())
    no_improve = 0
    patience   = config.get('p6_patience', 10)

    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [],
        'val_acc': [], 'ema_val_acc': [], 'lr': [], 'phase': [],
        'focal_loss': [], 'kd_loss': [], 'real_val_acc': [],
    }

    val_name = globals().get('VAL_SOURCE', 'old')
    print(f'  Train  : {len(trn_samples):,} | Val ({val_name}): {len(val_samples):,}')
    if real_monitor_samples:
        print(f'  real_monitor: {len(real_monitor_samples)} ảnh (LOG only)')
    print(f'  Base checkpoint val_acc: {base_best:.4f}')
    print(f'  Checkpoint selection: {selection_metric} (ưu tiên real-world accuracy)')
    print(f'  Early stop patience: {patience} epochs')

    # Đánh giá real_val ngay từ đầu để có baseline
    baseline_real_acc = 0.0
    if real_val_dl:
        _, baseline_real_acc, per_cls_real = validate(model, real_val_dl, criterion, device)
        print(f'  Baseline real_val acc: {baseline_real_acc:.4f}')
        print('  Per-class real baseline: ' +
              ' | '.join(f'{k[:6]}:{v:.2f}' for k, v in per_cls_real.items()))
    print()

    for ep in range(1, epochs + 1):
        t0 = time.time()

        # [v3] KD-aware training
        tl, tf, tkd, ta = train_one_epoch_kd(
            model, teacher, trn_dl, optimizer,
            criterion, kd_criterion, device,
            kd_weight=kd_w,
            mixup=config.get('p6_mixup', 0.1),
            scaler=scaler, epoch=ep,
            clip_grad=clip_g              # [v3] 1.0
        )
        ema.update(model, ep)

        # Validate trên validation source đã chuẩn bị ở Cell 7 (old hoặc real_monitor)
        vl, va, per_cls = validate(model, val_dl, criterion, device)
        _, ea, _        = validate(ema.ema, val_dl, criterion, device)

        # [v3] Real-world validation: dùng làm tiêu chí chọn checkpoint chính
        real_acc = real_ema_acc = 0.0
        real_eff_acc = 0.0
        real_src = 'none'
        if real_val_dl:
            _, real_acc, per_cls_real = validate(model, real_val_dl, criterion, device)
            _, real_ema_acc, _ = validate(ema.ema, real_val_dl, criterion, device)
            real_eff_acc = max(real_acc, real_ema_acc)
            real_src = 'EMA' if real_ema_acc > real_acc else 'model'

        scheduler.step()
        lr  = optimizer.param_groups[-1]['lr']
        dt  = time.time() - t0
        eff_acc = max(va, ea)
        eff_src = 'EMA' if ea > va else 'model'

        print(f"  P6v3[{ep:02d}/{epochs}] "
              f"L:{tl:.3f} A:{ta:.3f} | "
              f"vL:{vl:.3f} vA:{va:.3f} eA:{ea:.3f} | "
              f"focal:{tf:.3f} kd:{tkd:.3f} | "
              f"realA:{real_acc:.3f} realE:{real_ema_acc:.3f} | "
              f"lr:{lr:.1e} {dt:.0f}s{gpu_mem_str()}")

        if ep % 3 == 0 or ep == epochs:
            print('  Per-cls (old): ' +
                  ' | '.join(f'{k[:6]}:{v:.2f}' for k, v in per_cls.items()))
            if real_val_dl:
                print('  Per-cls (real): ' +
                      ' | '.join(f'{k[:6]}:{v:.2f}' for k, v in per_cls_real.items()))

        _track(history, tl, ta, vl, va, ea, 6, lr,
               focal=tf, kd=tkd,
               real_acc=real_eff_acc if real_val_dl else None)

        if real_val_dl and selection_metric == 'real':
            current_score = real_eff_acc
            score_desc = f'real={real_eff_acc:.4f}'
            state_to_save = ema.ema.state_dict() if real_src == 'EMA' else model.state_dict()
            save_src = f'real_{real_src}'
        elif real_val_dl and selection_metric == 'hybrid':
            current_score = real_score_w * real_eff_acc + (1.0 - real_score_w) * eff_acc
            score_desc = f'hybrid={current_score:.4f} old={eff_acc:.4f} real={real_eff_acc:.4f}'
            state_to_save = ema.ema.state_dict() if real_src == 'EMA' else model.state_dict()
            save_src = f'hybrid_{real_src}'
        else:
            current_score = eff_acc
            score_desc = f'old={eff_acc:.4f}'
            state_to_save = ema.ema.state_dict() if ea > va else model.state_dict()
            save_src = f'old_{eff_src}'

        # Early stop + checkpoint theo mục tiêu deploy: real_val_acc cao nhất.
        if current_score > best_score:
            best_score = current_score
            best_acc   = eff_acc
            best_real  = real_eff_acc
            best_epoch = ep
            best_src   = save_src
            best_w     = copy.deepcopy(state_to_save)
            torch.save({
                'epoch'            : ep,
                'model_state'      : best_w,
                'selection_metric' : selection_metric,
                'score'            : best_score,
                'val_acc'          : best_acc,
                'real_val_acc'     : best_real,
                'real_model_acc'   : real_acc,
                'real_ema_acc'     : real_ema_acc,
                'classes'          : CLASSES,
                'img_size'         : img_size,
                'agc_target'       : config['agc_target'],
                'agc_gamma_min'    : config['agc_gamma_min'],
                'agc_gamma_max'    : config['agc_gamma_max'],
                'kd_temperature'   : kd_T,
                'kd_weight'        : kd_w,
                'source'           : f'phase6_v3_{save_src}',
                'base_acc'         : base_best,
                'config'           : config,
            }, ckpt_p6)
            print(f'  >> Saved P6v3 best: score={best_score:.4f} ({save_src})  '
                  f'{score_desc}  ckpt={ckpt_p6}')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\n  >> Early stop ep{ep} (no score improve {no_improve}/{patience})')
                break

        cleanup_gpu()

    # ── Finalize ───────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  Phase 6 v3 DONE")
    print(f"  Base checkpoint val_acc : {base_best:.4f}")
    print(f"  Best selection score   : {best_score:.4f}  epoch={best_epoch}  source={best_src}")
    print(f"  Best old val acc       : {best_acc:.4f}")
    print(f"  Best real val acc      : {best_real:.4f}")
    if real_val_dl:
        print(f"  real_val baseline: {baseline_real_acc:.4f}")
        last_real = history['real_val_acc'][-1] if history['real_val_acc'] else 0
        real_direction = '↑' if last_real >= baseline_real_acc else '↓'
        print(f"  real_val final   : {last_real:.4f}  "
              f"({real_direction}{abs(last_real - baseline_real_acc)*100:.2f}% domain shift)")
    print(f"  Checkpoint .pth      : {ckpt_p6}")

    model.load_state_dict(best_w)

    print(f'\n  Final evaluation ({val_name})...')
    _, _, _, ap, al = validate(model, val_dl, criterion, device, return_preds=True)
    plot_confusion_matrix(ap, al, out_dir, suffix='_p6v3')
    plot_history(history, out_dir)

    if real_val_dl and best_real <= baseline_real_acc:
        print('\n  [WARN] Phase 6 v3 KHÔNG cải thiện real_val. Kiểm tra:')
        print('    1. DataRealTest đủ ảnh? (>30 ảnh/class tối thiểu)')
        print('    1b. Nếu có thể, khôi phục OLD dataset để val_old ổn định hơn.')
        print('    2. Label có bị sai? (xem confusion matrix)')
        print('    3. KD weight quá cao? Thử giảm p6_kd_weight về 0.2')
        print('    4. Thử tăng p6_epochs hoặc thu thập thêm data.')

    del trn_dl, val_dl
    if real_val_dl: del real_val_dl
    cleanup_gpu()
    return model, best_score


# ── RUN ──────────────────────────────────────────────────────────────────────
seed_everything(CONFIG['seed'])
p6_model, p6_acc = phase6_finetune_v3(
    trn_samples, val_samples, real_monitor_samples,
    config=CONFIG
)
print(f'\nPhase 6 v3 best selection score: {p6_acc:.4f}')



  PHASE 6 v3: REAL-WORLD FINE-TUNING + KNOWLEDGE DISTILLATION
  Device: mps  |  Epochs: 60  |  Batch: 8
  LR: head=2e-05  backbone=6e-07
  KD: T=4.0  weight=0.3  (focal_weight=0.7)
  clip_grad=1.0  EMA_decay=0.9995
  selection=real  real_weight=0.8

  [OK] Student loaded: outputs_v2/best_model_v2.pth
       val_acc=0.9871  source=phase6_realworld_model
  [OK] Teacher (frozen Phase 5) ready.
       Teacher eval mode — no gradients.
Dataloaders ready.
  real_monitor: 170 ảnh (real holdout, LOG only)
  Train  : 34,305 | Val (old): 3,803
  real_monitor: 170 ảnh (LOG only)
  Base checkpoint val_acc: 0.9871
  Checkpoint selection: real (ưu tiên real-world accuracy)
  Early stop patience: 20 epochs
  Baseline real_val acc: 0.8176
  Per-class real baseline: Batter:0.98 | Biolog:0.88 | Genera:0.45 | Glass:0.60 | Metal:0.78 | Paper_:0.77 | Plasti:0.85



  P6v3[01/60] L:0.520 A:0.684 | vL:0.125 vA:0.924 eA:0.918 | focal:0.497 kd:0.573 | realA:0.871 realE:0.865 | lr:2.0e-05 1134s
  >> Saved P6v3 best: score=0.8706 (real_model)  real=0.8706  ckpt=./outputs_v3/best_model_p6_v3.pth


  P6v3[02/60] L:0.463 A:0.696 | vL:0.104 vA:0.933 eA:0.928 | focal:0.422 kd:0.558 | realA:0.882 realE:0.882 | lr:2.0e-05 1133s
  >> Saved P6v3 best: score=0.8824 (real_model)  real=0.8824  ckpt=./outputs_v3/best_model_p6_v3.pth


  P6v3[03/60] L:0.428 A:0.725 | vL:0.099 vA:0.938 eA:0.934 | focal:0.375 kd:0.552 | realA:0.882 realE:0.882 | lr:2.0e-05 1118s
  Per-cls (old): Batter:0.97 | Biolog:1.00 | Genera:0.99 | Glass:0.98 | Metal:0.99 | Paper_:0.98 | Plasti:0.73
  Per-cls (real): Batter:0.98 | Biolog:0.88 | Genera:0.73 | Glass:1.00 | Metal:0.83 | Paper_:0.77 | Plasti:0.93


  P6v3[04/60] L:0.418 A:0.735 | vL:0.081 vA:0.948 eA:0.943 | focal:0.362 kd:0.547 | realA:0.894 realE:0.894 | lr:2.0e-05 2919s
  >> Saved P6v3 best: score=0.8941 (real_model)  real=0.8941  ckpt=./outputs_v3/best_model_p6_v3.pth


...

  P6v3[50/60] L:0.316 A:0.811 | vL:0.051 vA:0.971 eA:0.971 | focal:0.227 kd:0.524 | realA:0.906 realE:0.906 | lr:1.5e-06 1121s


  P6v3[51/60] L:0.313 A:0.808 | vL:0.054 vA:0.971 eA:0.971 | focal:0.222 kd:0.524 | realA:0.912 realE:0.906 | lr:1.3e-06 1131s
  Per-cls (old): Batter:0.99 | Biolog:1.00 | Genera:0.99 | Glass:0.98 | Metal:0.99 | Paper_:0.98 | Plasti:0.90
  Per-cls (real): Batter:1.00 | Biolog:0.88 | Genera:0.82 | Glass:1.00 | Metal:0.83 | Paper_:0.83 | Plasti:0.93


  P6v3[52/60] L:0.316 A:0.810 | vL:0.048 vA:0.975 eA:0.975 | focal:0.225 kd:0.527 | realA:0.900 realE:0.900 | lr:1.1e-06 1115s


  P6v3[53/60] L:0.310 A:0.807 | vL:0.056 vA:0.967 eA:0.967 | focal:0.219 kd:0.525 | realA:0.918 realE:0.918 | lr:8.6e-07 1126s


  P6v3[54/60] L:0.311 A:0.814 | vL:0.054 vA:0.972 eA:0.971 | focal:0.220 kd:0.523 | realA:0.912 realE:0.912 | lr:6.8e-07 1132s
  Per-cls (old): Batter:0.98 | Biolog:1.00 | Genera:0.98 | Glass:0.98 | Metal:0.99 | Paper_:0.98 | Plasti:0.91
  Per-cls (real): Batter:0.98 | Biolog:0.88 | Genera:0.86 | Glass:1.00 | Metal:0.83 | Paper_:0.83 | Plasti:0.93

  >> Early stop ep54 (no score improve 20/20)

  Phase 6 v3 DONE
  Base checkpoint val_acc : 0.9871
  Best selection score   : 0.9176  epoch=34  source=real_model
  Best old val acc       : 0.9663
  Best real val acc      : 0.9176
  real_val baseline: 0.8176
  real_val final   : 0.9118  (↑9.41% domain shift)
  Checkpoint .pth      : ./outputs_v3/best_model_p6_v3.pth

  Final evaluation (old)...
  >> Confusion: ./outputs_v3/confusion_matrix_p6v3.png
                 precision    recall  f1-score   support

        Battery       0.98      0.98      0.98       158
     Biological       0.98      1.00      0.99       564
  General_Waste       0.

In [9]:
# ============================================================
# CELL 9 - EXPORT ONNX v3  (Phase 6 v3 checkpoint)
# ============================================================
%pip install onnxscript -q

def export_onnx_v3(model, config=CONFIG, img_size=None, suffix='_v3', ckpt_path=None):
    """Export Phase 6 v3 best checkpoint → ONNX + model_meta.json."""
    img_size  = img_size or config.get('p6_img_size', 384)
    out_dir   = config.get('output_dir', './outputs_v3')
    os.makedirs(out_dir, exist_ok=True)

    export_model = copy.deepcopy(model).cpu().eval()
    ckpt_meta = {}
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        export_model.load_state_dict(ck.get('model_state', ck), strict=True)
        ckpt_meta = ck if isinstance(ck, dict) else {}
        print(f'  >> Loaded best .pth for export: {ckpt_path}')
    cpu_model = export_model
    dummy     = torch.randn(1, 3, img_size, img_size)
    onnx_name = f'waste7_detector{suffix}.onnx'
    out_path  = os.path.join(out_dir, onnx_name)

    torch.onnx.export(
        cpu_model, (dummy,), out_path,
        input_names=['image'], output_names=['logits', 'objectness'],
        opset_version=18,
        do_constant_folding=True,
        dynamic_axes={
            'image'      : {0: 'batch'},
            'logits'     : {0: 'batch'},
            'objectness' : {0: 'batch'},
        },
        external_data=False,
    )
    del cpu_model, dummy; gc.collect()
    print(f'  >> ONNX v3  : {out_path}')

    # model_meta.json — testPC.py đọc file này để cấu hình AGC
    meta = {
        'classes'        : CLASSES,
        'img_size'       : img_size,
        'agc_target'     : config['agc_target'],
        'agc_gamma_min'  : config['agc_gamma_min'],
        'agc_gamma_max'  : config['agc_gamma_max'],
        'phase'          : 'phase6_v3_realworld_kd',
        'selection_metric': ckpt_meta.get('selection_metric', config.get('p6_selection_metric', 'real')),
        'score'          : ckpt_meta.get('score'),
        'val_acc'        : ckpt_meta.get('val_acc'),
        'real_val_acc'   : ckpt_meta.get('real_val_acc'),
        'kd_temperature' : config.get('p6_kd_temperature', 4.0),
        'kd_weight'      : config.get('p6_kd_weight', 0.3),
        'clip_grad'      : config.get('p6_clip_grad', 1.0),
    }
    meta_path = os.path.join(out_dir, 'model_meta_v3.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  >> Meta     : {meta_path}')
    print(f'     AGC: target={config["agc_target"]}  '
          f'clip=[{config["agc_gamma_min"]}, {config["agc_gamma_max"]}]')
    print(f'     KD: T={meta["kd_temperature"]}  weight={meta["kd_weight"]}')
    print(f'     Selection: {meta["selection_metric"]}  score={meta["score"]}  real={meta["real_val_acc"]}')
    print(f'\n  Deploy:')
    print(f'    1. Copy {onnx_name} + model_meta_v3.json → thư mục testPC.py')
    print(f'    2. Đổi ONNX_PATH = "{onnx_name}" trong testPC.py')
    print(f'    3. Đổi META_PATH = "model_meta_v3.json" trong testPC.py')
    return out_path


# Chạy export từ best .pth của Phase 6 v3
best_pth_v3 = os.path.join(CONFIG.get('output_dir', './outputs_v3'), 'best_model_p6_v3.pth')
export_onnx_v3(p6_model, CONFIG, ckpt_path=best_pth_v3)


Note: you may need to restart the kernel to use updated packages.
  >> Loaded best .pth for export: ./outputs_v3/best_model_p6_v3.pth
[torch.onnx] Obtain model graph for `WasteDetector([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `WasteDetector([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
  >> ONNX v3  : ./outputs_v3/waste7_detector_v3.onnx
  >> Meta     : ./outputs_v3/model_meta_v3.json
     AGC: target=128  clip=[0.4, 3.0]
     KD: T=4.0  weight=0.3
     Selection: real  score=0.9176470588235294  real=0.9176470588235294

  Deploy:
    1. Copy waste7_detector_v3.onnx + model_meta_v3.json → thư mục testPC.py
    2. Đổi ONNX_PATH = "waste7_detector_v3.onnx" trong testPC.py
    3. Đổi 

'./outputs_v3/waste7_detector_v3.onnx'